In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap
import warnings 
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

from config.constants import PROJECT_ROOT

edges_path = str(PROJECT_ROOT / "saved_data" / "cohorts" / "DTB" / "new" / "selected_edges_DTB_all.csv")
save_dir = str(PROJECT_ROOT / "saved_data" / "plots" / "DTB") + "/"
cohorts_dir = str(PROJECT_ROOT / "saved_data" / "cohorts" / "DTB" / "new")


os.makedirs(save_dir, exist_ok=True)

In [ ]:
sel_edges = pd.read_csv(edges_path)
sel_edges = sel_edges[(sel_edges.n_neg > 50) & (sel_edges.n_pos > 10)]


sel_edges["AGE_AT_DISEASE_years"] = sel_edges["AGE_AT_DISEASE"] / 365.25
sel_edges["female_rate"] = sel_edges["female_counts"] / sel_edges["counts"]
sel_edges["female_rate_cohort"] = sel_edges["female_counts_cohort"] / sel_edges["counts_cohort"]
sel_edges["death_rate"] = sel_edges["death_counts"] / sel_edges["counts"]
sel_edges["death_rate_cohort"] = sel_edges["death_counts_cohort"] / sel_edges["counts_cohort"]
sel_edges["log_n"] = np.log10(sel_edges["counts_cohort"].clip(lower=1))

plot_df = sel_edges.sort_values("counts_cohort")

comparisons = [
    ("counts", "counts_cohort", "Population size", True),
    ("AGE_AT_DISEASE_years", "AGE_AT_DISEASE_cohort", "Age at disease (years)", False),
    ("female_rate", "female_rate_cohort", "Female rate", False),
    ("death_rate", "death_rate_cohort", "Death rate", False),
    ("CODE_DIFF_DAYS", "CODE_DIFF_DAYS_cohort", "Days D1\u2192D2", True),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
cmap = plt.cm.Blues

for ax, (paper_col, cohort_col, title, log_scale) in zip(axes, comparisons):
    d = plot_df[[paper_col, cohort_col, "log_n"]].dropna()
    sc = ax.scatter(d[paper_col], d[cohort_col], c=d["log_n"], cmap=cmap,
                     s=14, alpha=0.6, vmin=0, vmax=sel_edges["log_n"].max(),
                     edgecolors="none")
    lo = min(d[paper_col].min(), d[cohort_col].min())
    hi = max(d[paper_col].max(), d[cohort_col].max())
    ax.plot([lo, hi], [lo, hi], color="gray", linestyle="--", linewidth=1)
    if log_scale:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.set_xlabel(f"{title} (DTB)")
    ax.set_ylabel(f"{title} (MIMIC)")
    ax.set_title(title)
    r = d[paper_col].corr(d[cohort_col])
    ax.annotate(f"r={r:.2f}  n={len(d)}", xy=(0.05, 0.92), xycoords="axes fraction")

axes[-1].axis("off")
cbar = fig.colorbar(sc, ax=axes[-1], fraction=0.6, aspect=15)
cbar.set_label("log10(cohort N)")

plt.tight_layout()
plt.savefig(save_dir + "paper_vs_cohort_comparison.png", dpi=150)
plt.show()


### Load initial cohorts and extract first admissions only

In [ ]:
sel_edges = sel_edges.drop(columns=["direction_yes_no","p5years","p.value.direction","n_neg","n_pos","log_n"])
sel_edges["cohort_name"] = sel_edges["D1"] + "-" + sel_edges["D2"]

In [ ]:
cohorts_dir = PROJECT_ROOT / "saved_data" / "cohorts" / "DTB" / "new"
cohort_files = sorted(cohorts_dir.glob("*.csv.gz"))
print(len(cohort_files))

rows = []
rows_first_adm = []
for f in cohort_files:
    cohort = f.name.removesuffix(".csv.gz")
    df = pd.read_csv(f, usecols=["subject_id", "admittime", "label"])

    n_pos = int(df["label"].sum())
    n_cohort = len(df)
    rows.append(dict(cohort_name=cohort, n_cohort=n_cohort, n_pos=n_pos, n_neg=n_cohort - n_pos))

    df_first = df.sort_values("admittime").drop_duplicates("subject_id", keep="first")
    n_pos_first = int(df_first["label"].sum())
    n_cohort_first = len(df_first)
    rows_first_adm.append(dict(cohort_name=cohort, n_cohort=n_cohort_first, n_pos=n_pos_first, n_neg=n_cohort_first - n_pos_first))

cohort_counts = pd.DataFrame(rows)
cohort_counts_first_adm = pd.DataFrame(rows_first_adm)


# Compute cohort specific characteristics

In [ ]:
# merge with edge characteristics
cohort_counts = cohort_counts.merge(sel_edges, on = "cohort_name")
cohort_counts_first_adm = cohort_counts_first_adm.merge(sel_edges, on = "cohort_name")

In [ ]:
def describe_cohort_characteristics(df):
    df["target_rate"] = df["n_pos"] / df["n_cohort"]
    df["log_target_rate"] = np.log(df["target_rate"])
    df["log_n_cohort"] = np.log(df["n_cohort"])
    df["log_RR"] = np.log(df["RR"])
    df["imbalance_ratio"] = df["n_neg"] / df["n_pos"]
    df["log_imb"] = np.log(df["imbalance_ratio"])
    icd_map = {
        "A": "infectious", "B": "infectious", "C": "neoplasms",
        "D": "blood", "E": "metabolic", "F": "mental",
        "G": "nervous", "H": "sensory", "I": "circulatory",
        "J": "respiratory", "K": "digestive", "L": "skin",
        "M": "musculoskeletal", "N": "genitourinary", "O": "pregnancy",
        "P": "perinatal", "Q": "congenital"
    }
    df["D1type"] = df["D1"].str[:1].map(icd_map)
    df["D2type"] = df["D2"].str[:1].map(icd_map)
    
    return df

In [ ]:
cohort_counts = describe_cohort_characteristics(cohort_counts)
cohort_counts_first_adm = describe_cohort_characteristics(cohort_counts_first_adm)

In [ ]:
cohort_counts_first_adm.columns

# Plot each characteristic

In [ ]:
feats = [
    "n_cohort", "n_pos", "n_neg", "RR",
    "AGE_AT_DISEASE", "counts", "female_counts", "CODE_DIFF_DAYS", "death_counts",
    "counts_cohort", "female_counts_cohort", "AGE_AT_DISEASE_cohort", "CODE_DIFF_DAYS_cohort", "death_counts_cohort",
    "AGE_AT_DISEASE_years", "female_rate", "female_rate_cohort", "death_rate", "death_rate_cohort",
    "target_rate", "log_target_rate", "log_n_cohort", "log_RR", "imbalance_ratio", "log_imb",
]

labels = [
    "Cohort Size", "N Positive", "N Negative", "Relative Risk",
    "Age at Disease (DTB, days)", "Population Size (DTB)", "Female Count (DTB)", "Days D1\u2192D2 (DTB)", "Death Count (DTB)",
    "Population Size (MIMIC)", "Female Count (MIMIC)", "Age at Disease (MIMIC, years)", "Days D1\u2192D2 (MIMIC)", "Death Count (MIMIC)",
    "Age at Disease (DTB, years)", "Female Rate (DTB)", "Female Rate (MIMIC)", "Death Rate (DTB)", "Death Rate (MIMIC)",
    "Positive Rate", "Log Positive Rate", "Log Cohort Size", "Log Relative Risk", "Imbalance Ratio", "Log Imbalance Ratio",
]

feat_labels = dict(zip(feats, labels))


feat_to_plot_1 = "log_imb"
#feat_to_plot_2 = "AGE_AT_DISEASE_cohort"

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.histplot(cohort_counts[feat_to_plot_1].dropna(), bins=40, ax=ax, color="#431C3A",
             edgecolor="white", linewidth=0.4, alpha = 0.5,stat="count", label="all admissions")

sns.histplot(cohort_counts_first_adm[feat_to_plot_1].dropna(), bins=40, ax=ax, color="#1C7A8A",
             edgecolor="white", linewidth=0.4, alpha = 0.5,stat="count", label="first admission")
ax.set_xlabel(feat_labels[feat_to_plot_1], fontsize=13)
ax.set_ylabel("Nr. of cohorts", fontsize=13)
ax.tick_params(labelsize=14)
ax.legend()
plt.tight_layout()
plt.savefig(save_dir + f"cohort_{feat_to_plot_1}_compare.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
hue_order = sorted(
    set(cohort_counts["D1type"].dropna().unique()) |
    set(cohort_counts_first_adm["D1type"].dropna().unique())
)
palette_dict = dict(zip(hue_order, sns.color_palette("husl", len(hue_order))))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

for ax, data, title in zip(
    axes,
    [cohort_counts, cohort_counts_first_adm],
    ["All admissions", "First admission only"],
):
    sns.scatterplot(
        data=data,
        x="log_n_cohort",
        y="log_target_rate",
        hue="D1type",
        hue_order=hue_order,
        palette=palette_dict,
        s=15,
        alpha=0.5,
        legend=False,
        ax=ax,
    )
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Log Cohort Size", fontsize=16)
    ax.tick_params(labelsize=11)

axes[0].set_ylabel("Log Positive Rate", fontsize=16)
axes[1].set_ylabel("")

handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=palette_dict[d], markersize=6) for d in hue_order]
fig.legend(
    handles, hue_order, title="D1 Group",
    bbox_to_anchor=(1.0, 0.5), loc="center left",
    fontsize=12, title_fontsize=14, frameon=False,
)

plt.tight_layout()
plt.savefig(save_dir + "cohort_size_vs_target_rate_D1cat_compare.png", dpi=300, bbox_inches="tight")
plt.show()


# How target rate and cohort sizes changes when we switch to first admission only?

In [ ]:
merged = cohort_counts.merge(cohort_counts_first_adm, on="cohort_name", suffixes=("_all", "_first"))
merged["cohort_size_ratio"] = merged["n_cohort_first"] / merged["n_cohort_all"]
merged["n_pos_ratio"] = merged["n_pos_first"] / merged["n_pos_all"]
merged["n_neg_ratio"] = merged["n_neg_first"] / merged["n_neg_all"]
merged["target_rate_delta"] = merged["target_rate_first"] - merged["target_rate_all"]

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes = axes.flatten()

for ax, col, title in zip(
    axes,
    ["cohort_size_ratio", "n_pos_ratio", "n_neg_ratio", "target_rate_delta"],
    ["Cohort size retained (first/all)", "n_pos retained (first/all)",
     "n_neg retained (first/all)", "Target rate change (first - all)"],
):
    ax.hist(merged[col].dropna(), bins=50, alpha = 0.7,color="#183265", edgecolor="white", linewidth=0.4)
    ax.axvline(merged[col].median(), color="firebrick", linestyle="--", linewidth=1.5,
               label=f"median={merged[col].median():.2f}")
    ax.set_xlabel(title, fontsize=16)
    ax.set_ylabel("Nr. of cohorts", fontsize=16)
    ax.legend()

plt.tight_layout()
plt.savefig(save_dir + "firstadm_shrinkage_distributions.png", dpi=150)
plt.show()


# Filter unreliable cohorts 

After stratified cross-validation, each test split should have enough positive and negative samples to reliably evaluate model performance.
Based on stratified cross fold validation we have an estimate of how many positive samples we will have in test/train
We filter cohorts to have at lest 10 positive and 10 negative samples in the test set -->  <span style="color:red">**To be checked later**</span>

In [ ]:
cohort_counts_first_adm.target_rate.describe()


In [ ]:
# since our cohorts are heavily right skewed we prefer 80 (64/16) /20 split not 80/10/10
test_frac = 0.2 
min_test_n = 10 # 5 fold cross validation we should have at least 50 positives in the cohort

In [ ]:
def add_reliability(df):
    df = df.copy()
    df["n_pos_test_estimate"] = df["n_pos"] * test_frac
    df["n_neg_test_estimate"] = df["n_neg"] * test_frac
    df["reliable"] = (df["n_pos_test_estimate"] >= min_test_n) & (df["n_neg_test_estimate"] >= min_test_n)
    return df

cohort_counts = add_reliability(cohort_counts)
cohort_counts_first_adm = add_reliability(cohort_counts_first_adm)

print("All admissions:", cohort_counts["reliable"].sum(), "/", len(cohort_counts))
print("First admission only:", cohort_counts_first_adm["reliable"].sum(), "/", len(cohort_counts_first_adm))


reliable_cohorts_initial = cohort_counts[cohort_counts.reliable == True]
reliable_cohorts_first_adms = cohort_counts_first_adm[cohort_counts_first_adm.reliable == True]

In [ ]:
threshold = min_test_n / test_frac

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True, sharey=True,
                          gridspec_kw={"wspace": 0.15})

for ax, df, name in zip(axes, [cohort_counts, cohort_counts_first_adm], ["All admissions", "First Admissions"]):
    sns.scatterplot(data=df, x="n_pos", y="n_neg", hue="reliable",
                     palette={True: "#2E7D32", False: "#B23B3B"},
                     s=18, alpha=0.6, edgecolor="none", ax=ax, legend=False)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axvline(threshold, color="black", linestyle="--", linewidth=1)
    ax.axhline(threshold, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("n_pos",fontsize = 16)
    ax.set_title(name,fontsize = 16)

    n_total = len(df)
    n_reliable = int(df["reliable"].sum())
    n_unreliable = n_total - n_reliable
    ax.text(0.02, 0.98, f"total: {n_total}\nreliable: {n_reliable}\nnot reliable: {n_unreliable}",
            transform=ax.transAxes, ha="left", va="top", fontsize=10,
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray", alpha=0.8))

axes[0].set_ylabel("n_neg",fontsize = 16)
axes[1].set_ylabel("n_neg",fontsize = 16)

handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color="#2E7D32", markersize=6),
    plt.Line2D([0], [0], marker="o", linestyle="", color="#B23B3B", markersize=6),
]
fig.legend(handles, ["True", "False"], title="reliable",  frameon=False)

plt.tight_layout()
plt.savefig(save_dir + "threshold_scatter_compare.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

for ax, data, title in zip(
    axes,
    [cohort_counts, cohort_counts_first_adm],
    ["All admissions", "First admission only"],
):
    sns.scatterplot(
        data=data,
        x="log_n_cohort",
        y="log_target_rate",
        hue="reliable",
        palette={True: "#2E7D32", False: "#B23B3B"},
        s=15,
        alpha=0.5,
        legend=False,
        ax=ax,
    )
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Log Cohort Size", fontsize=16)
    ax.tick_params(labelsize=11)

    n_total = len(data)
    n_reliable = int(data["reliable"].sum())
    ax.text(0.02, 0.98, f"total: {n_total}\nreliable: {n_reliable}\nnot reliable: {n_total - n_reliable}",
            transform=ax.transAxes, ha="left", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray", alpha=0.8))

axes[0].set_ylabel("Log Positive Rate", fontsize=16)
axes[1].set_ylabel("")

handles = [
    plt.Line2D([0], [0], marker="o", linestyle="", color="#2E7D32", markersize=6),
    plt.Line2D([0], [0], marker="o", linestyle="", color="#B23B3B", markersize=6),
]
fig.legend(handles, ["True", "False"], title="reliable",
           bbox_to_anchor=(1.0, 0.5), loc="center left", fontsize=12, title_fontsize=14, frameon=False)

plt.tight_layout()
plt.savefig(save_dir + "cohort_size_vs_target_rate_reliable_compare.png", dpi=300, bbox_inches="tight")
plt.show()


# Save Cohorts Summary

In [ ]:
cohort_counts.to_csv(cohorts_dir / "initial_cohorts_summary.csv", index=False)
cohort_counts_first_adm.to_csv(cohorts_dir / "first_adms_cohorts_summary.csv", index=False)
reliable_cohorts_initial["cohort_name"].to_csv(cohorts_dir / "reliable_cohorts_init.csv", index=False, header=False)
reliable_cohorts_first_adms["cohort_name"].to_csv(cohorts_dir / "reliable_cohorts_first_adms.csv", index=False, header=False)

---

----

In [ ]:
cohorts_init = pd.read_csv(cohorts_dir / "initial_cohorts_summary.csv")
cohorts_first_adms = pd.read_csv(cohorts_dir / "first_adms_cohorts_summary.csv")
reliable_cohorts_init = pd.read_csv(cohorts_dir / "reliable_cohorts_init.csv", header=None, names=["cohort_name"])
reliable_cohorts_first_adms = pd.read_csv(cohorts_dir / "reliable_cohorts_first_adms.csv", header=None, names=["cohort_name"])
cohorts_first_adms_reliable = cohorts_first_adms[cohorts_first_adms["cohort_name"].isin(reliable_cohorts_first_adms["cohort_name"])]
cohorts_init_reliable = cohort_counts[cohort_counts["cohort_name"].isin(reliable_cohorts_init["cohort_name"])]


In [ ]:
rep_cohorts = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/DTB/new/representative_cohorts_df.csv").cohort_name

In [ ]:
overlap = set(rep_cohorts) & set(reliable_cohorts_first_adms.cohort_name)
len(overlap)

In [ ]:
def load_results(cohorts, models, prefixes,results_dir, fold=None):

    folds = [fold] if fold is not None else range(5)
    rows = []
    for cohort in cohorts:
        for model in models:
            prefix = next((p for p in prefixes if (Path(f"../saved_data/{results_dir}/{cohort}/{model}") / p).exists()),None)
            if prefix is None:
                print("no prefix")
                continue
            for f in folds:
                file = (Path(f"../saved_data/{results_dir}/{cohort}/{model}/{prefix}")
                        / f"fold_{f}" / "agg_int_24" / "impute_fill" / "variant_VMD" / "results_final.csv")
                if not file.exists():
                    print(f"{cohort} - fold_{f} doesn't exist")
                    continue
                output = pd.read_csv(file)
                output["cohort_name"] = cohort
                output["model"] = model
                output["prefix"] = prefix
                output["fold"] = f
                rows.append(output)
                
    results_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    results_df = results_df[["cohort_name","model","prefix","fold","auroc", "auprc", "f1"]]
    
    for col in ["auroc", "auprc", "f1"]:
        results_df[col] = pd.to_numeric(results_df[col], errors="coerce")

    '''for col, new_col in [("auroc", "roc_avg"), ("auprc", "pr_avg"), ("f1", "f1_avg")]:
        results_df[new_col] = results_df.groupby(["cohort_name", "model"])[col].transform("mean") # average of folds'''
        
    #results_df = results_df[["cohort_name", "model", "roc_avg", "pr_avg", "f1_avg"]].drop_duplicates(["cohort_name", "model"])
    return results_df

In [ ]:
results_dir = "results_initial"

models = ["random_forest"]
# Three runs for representative cohorts:
# 190526 (11 cohorts) — no overlap with 200526 (primary run)
# 200526 (89 cohorts) — same run (rest of the cohorts)
# 030626 (7 cohorts: I70-A49, J38-J06, K59-J69, L02-A49, M62-M35, N81-N99, N92-N89) -- rerun because of the crash due to no positive in test
prefixes = ["200526", "190526"] 
folds = 0

results_initial = load_results(overlap, models, prefixes,results_dir, folds)

In [ ]:
results = results_initial.merge(cohorts_first_adms_reliable,on="cohort_name")

In [ ]:
results

In [ ]:
perf_cols = ["auroc", "auprc", "f1"]
char_cols = ["log_n_cohort", "target_rate", "log_RR", "log_imb",
             "AGE_AT_DISEASE_cohort", "female_rate_cohort", "death_rate_cohort"]

df_agg = results.groupby(["cohort_name", "model"] + char_cols + ["D1type", "D2type"], as_index=False)[perf_cols].mean()


In [ ]:
df_agg["auprc_lift"] = df_agg["auprc"] - df_agg["target_rate"]

perf_cols_adj = ["auroc", "auprc_lift", "f1"]
corr_adj = df_agg[char_cols + perf_cols_adj].corr(method="spearman")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_adj.loc[char_cols, perf_cols_adj], annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Characteristic vs. performance (baseline-adjusted)")
plt.tight_layout()
plt.savefig(save_dir + "char_vs_perf_corr_adjusted.png", dpi=150)
plt.show()

